## Section 10: Recommendations for Future Development

### 1. Add Live PDF Preview
**Current Gap**: Users can't see final PDF before download

**Implementation**:
```
POST /api/certificates/preview
├─ Accept JSON (not saved to DB)
├─ Generate PDF in-memory
├─ Return as blob
└─ Display in modal with iframe or PdfPageCanvas

UI Enhancement:
├─ Add "Preview" button on form (alongside Submit)
├─ Open modal showing PDF
└─ Let users "Make Changes" or "Save & Download"
```

### 2. Fully Integrate Dynamic Templates
**Current Gap**: Template system exists in DB but not fully used

**Enhancement**:
```
1. Expose template editor UI (admin for team)
2. Allow sections to be toggled visible/hidden
3. Support custom styling per section
4. Store section-level templates in DB
5. Update PDF generator to respect section visibility
6. Add versioning + rollback
```

### 3. Field Mapping Visibility Tool
**Opportunity**: Help users understand form → PDF mapping

**Implementation**:
- Create mapping registry (JSON file mapping form fields → PDF sections)
- Add inline help tooltips: "This field appears as X in the PDF"
- Document field transformations per certificate type

### 4. Cached PDF Generation (Performance)
**Current**: PDFs regenerated every download

**Option**:
- Add optional PDF caching: store generated PDF after first generation
- Invalidate cache on certificate update
- Add "Regenerate PDF" button for manual refresh
- Useful for large batches

### 5. Extend Preview to Pre-Save
**Use Case**: Let users preview before creating certificate

**Implementation**:
- Add temporary certificate endpoint that doesn't require save
- Generate preview PDF for partial form data
- Helpful for complex forms like EICR

### 6. Export Settings / Template Config
**Documentation**:
- Document all template override options
- Provide example template JSON configs per company branding
- Create template migration tool for team transitions

## Section 9: Summary of Findings

### ✅ PDF Generation Infrastructure
- **Main Engine**: `lib/pdf/generator.ts` (jsPDF-based, 1400+ lines)
- **Supported Types**: BS5839-1, BS5839-6, BS5266, FIRE_EXTINGUISHER, DRY_RISER, EICR
- **EICR Generator**: Specialized multi-page layout with 8-page document support
- **Rendering**: On-demand from API `/api/certificates/[id]/pdf`
- **Colors**: Fully template-configurable with fallback defaults
- **Fonts**: Helvetica + Symbol for special characters (Δ, Ω)

### 📋 Template System
- **Type**: Hybrid (hardcoded default + optional database override)
- **Default**: All layouts hardcoded in generator functions
- **Optional DB**: `certificateTemplates` table stores color/font overrides per team per certificate type
- **Seed Script**: `scripts/seed-eicr-template.ts` demonstrates dynamic template definitions (structure-first, not yet fully integrated)
- **Integration**: PDF API loads template if available, otherwise uses hardcoded defaults

### 🔄 Form → PDF Data Flow
1. User fills form (one of 6 certificate types)
2. Form submit → `createCertificate()` server action
3. Data split into main fields + formData JSON + certificateItems array
4. Stored in PostgreSQL: `certificates` (main + formData JSON), `certificateItems` (one per item)
5. On download: `/api/certificates/[id]/pdf` fetches from DB and generates PDF on-demand
6. Transformations applied: date formatting, observation codes → colors, auto text wrapping, special chars

### 📐 EICR Certificate Specifics
- **8-page document** with specialized layout
- **Sections**: Report title, Client details, Reason, Installation details, Extent/Limitations, Assessment, Observations, Declaration
- **Observations**: C1 (red=danger), C2 (orange=risky), C3 (blue=improved), FI (purple=investigate)
- **Special Features**:
  - Greek letter support (Δ for delta, Ω for ohms)
  - Company branding (trading title, email in header/footer)
  - BS 7671 compliance formatting
  - Automatic section breaks

### 📊 Certificate Type Fields at a Glance

| Type | Form Fields | PDF Pages | Key Data | Notes |
|------|-------------|-----------|----------|-------|
| BS5839-1 | 14+ | 2-4 | System type, zones, devices | L1/L2/L3 systems |
| BS5839-6 | 13+ | 2-4 | Smoke/heat detectors, grade | Domestic properties |
| BS5266 | 10+ | 2-3 | Lamp count, battery blocks | Emergency lighting |
| FIRE_EXTINGUISHER | 7+ | 1-2 | Inventory, service dates | Portable units |
| DRY_RISER | 9+ | 2-3 | Building height, inlets | Water supply systems |
| EICR | 35+ | 8 | Wiring age, observations, earthing | Electrical installation |

### ❌ No Live Preview Currently
- No preview window before save
- No preview before download
- Users must download PDF to see final result
- `PdfPageCanvas` component exists (used by Report Disseminator) but not integrated into certificate flow
- Opportunity: Add `/api/certificates/preview` endpoint + modal preview

### 📦 Database Schema
- `certificates` (main records with formData JSON column)
- `certificateItems` (defects, recommendations, status)
- `certificateTemplates` (optional color/font overrides)
- `customers` (linked for site info auto-population)
- All forms auto-populate from selected customer

### 🎨 Color Customization
**Default Palette**:
- Navy: [26, 58, 92] — headers, primary elements
- Gold: [255, 193, 7] — accents, certificate number boxes
- Green: [40, 167, 69] — success / "no defects" sections
- Red: [220, 53, 69] — defects, C1 observations
- Orange: [255, 140, 0] — C2 observations
- Purple: [100, 55, 155] — FI observations

**Customizable via Template**:
- primary, secondary, accent, background, text colors
- Heading and body fonts
- Margins and spacing

### 🔗 Key File Dependencies
- **PDF generation route**: `app/api/certificates/[id]/pdf/route.ts` ← `lib/pdf/generator.ts`
- **Download button**: `components/DownloadPDFButton.tsx` → calls `/api/certificates/[id]/pdf`
- **Form pages**: `app/(dashboard)/certificates/new/[type]/page.tsx` → `actions.ts` → database
- **Template lookup**: `app/api/certificates/[id]/pdf/route.ts` → `certificateTemplates` table
- **Preview infrastructure**: `components/disseminator/PdfPageCanvas.tsx` (independent, not integrated)

## Section 8: Complete Form → PDF Architecture Diagram

```
╔═══════════════════════════════════════════════════════════════════════════╗
║                    AI CERTIFY FORM → PDF PIPELINE                         ║
╚═══════════════════════════════════════════════════════════════════════════╝

┌─────────────────────────────────────────────────────────────────────────┐
│ LAYER 1: FRONTEND (User Entry & Display)                               │
└─────────────────────────────────────────────────────────────────────────┘

  [Certificate Form Page]
  /certificates/new/[certType]/page.tsx
  ├─ Certificate metadata (number, dates, inspector)
  ├─ Customer selection → auto-populate fields
  ├─ Certificate items entry (defects, recommendations)
  └─ Type-specific formData fields
  
         ↓ FormData serialization
         
  [Form Submit Handler]
  onSubmit → createCertificate(formData)

┌─────────────────────────────────────────────────────────────────────────┐
│ LAYER 2: SERVER ACTION (Data Normalization)                            │
└─────────────────────────────────────────────────────────────────────────┘

  app/(dashboard)/actions.ts
  ├─ Parse FormData into typed object
  ├─ Extract main certificate fields
  ├─ Collect remaining fields into formData: Record<string, any>
  ├─ Serialize certificateItems array JSON
  └─ Validate required fields
  
         ↓ Validated certificate object

┌─────────────────────────────────────────────────────────────────────────┐
│ LAYER 3: DATABASE (Persistent Storage)                                 │
└─────────────────────────────────────────────────────────────────────────┘

  PostgreSQL Database
  ├─ certificates (main record)
  │  ├─ id, certificateNumber, certificateType, status
  │  ├─ siteName, siteAddress
  │  ├─ inspectionDate, nextInspectionDate
  │  ├─ formData (JSON column) ← all extra fields
  │  └─ createdAt, updatedAt
  │
  ├─ certificateItems (one per item)
  │  ├─ itemType, location, description, status
  │  ├─ defects, recommendations
  │  └─ certificateId (FK)
  │
  ├─ customers (linked)
  │  └─ name, email, phone, address, postcode, contactPerson
  │
  └─ certificateTemplates (optional color override)
     ├─ colors { primary, secondary, accent, background, text }
     ├─ fonts { heading, body, sizes }
     └─ layout { margins, spacing }

         ↓ User downloads PDF

┌─────────────────────────────────────────────────────────────────────────┐
│ LAYER 4: PDF GENERATION API (On-Demand Rendering)                      │
└─────────────────────────────────────────────────────────────────────────┘

  GET /api/certificates/[id]/pdf
  
  Step 1: Fetch from DB
  ├─ SELECT * FROM certificates WHERE id = ?
  ├─ SELECT * FROM customers WHERE id = certificates.customerId
  ├─ SELECT * FROM certificateItems WHERE certificateId = ?
  └─ SELECT * FROM certificateTemplates WHERE certificateType = ? (optional)
  
  Step 2: Construct CertificateData interface
  ├─ Merge DB records
  ├─ Load templateConfig colors/fonts
  └─ Prepare items array
  
  Step 3: Route to appropriate generator
  ├─ if type === 'EICR' → generateEICRPDF(data)
  └─ else → generateCertificatePDF(data)
  
         ↓ PDF generation engine

┌─────────────────────────────────────────────────────────────────────────┐
│ LAYER 5: PDF GENERATION (lib/pdf/generator.ts)                         │
└─────────────────────────────────────────────────────────────────────────┘

  generateCertificatePDF() or generateEICRPDF()
  
  Input: CertificateData {
    id, certificateNumber, certificateType, siteName, siteAddress,
    inspectionDate, nextInspectionDate, inspectorName, status, formData,
    templateConfig: { colors, fonts, layout },
    customer: { name, email, phone, address, postcode, contactPerson },
    items: [{ itemType, location, description, status, defects, recommendations }]
  }
  
  Processing:
  ├─ Initialize jsPDF (A4 page, margins)
  ├─ Layout header (company branding, report title)
  ├─ Render sections (1-9) based on certificate type
  │  ├─ Site Details
  │  ├─ System/Equipment Details (from formData)
  │  ├─ Inspection Details
  │  ├─ Equipment Items Tested (from certificateItems table)
  │  ├─ Defects & Recommendations (color-coded by severity)
  │  ├─ Certification Statement
  │  └─ Signatures
  ├─ Auto page breaks when content exceeds page
  ├─ Apply template colors override (if available)
  └─ Generate Uint8Array
  
  Field Transformations:
  ├─ formatDate(ISO) → "DD Month YYYY"
  ├─ Observation code (C1) → Color box + label
  ├─ Boolean (Yes/No) → Text display
  ├─ Multi-line text → pdf.splitTextToSize()
  └─ Special chars (Δ, Ω) → Symbol font swap

  EICR-Specific:
  ├─ 8-page specialized layout
  ├─ Observations mapped to C1/C2/C3/FI codes with colors
  ├─ Special electrical symbols support
  ├─ Company header footer with trading title
  └─ BS 7671 compliance formatting

  Output: Uint8Array (PDF bytes)

┌─────────────────────────────────────────────────────────────────────────┐
│ LAYER 6: HTTP RESPONSE → BROWSER DOWNLOAD                              │
└─────────────────────────────────────────────────────────────────────────┘

  Response Headers:
  ├─ Content-Type: application/pdf
  ├─ Content-Length: [bytes size]
  └─ Content-Disposition: attachment; filename="certificate-[number].pdf"
  
         ↓ Browser downloads file
  
  [Downloaded PDF File]


╔═══════════════════════════════════════════════════════════════════════════╗
║ PREVIEW INFRASTRUCTURE (Currently Unused for Certificates)               ║
╚═══════════════════════════════════════════════════════════════════════════╝

[No Live Preview Available]
- Users must save certificate first
- Preview only available via full PDF download
- Report Disseminator has separate PDF canvas preview 
  (components/disseminator/PdfPageCanvas.tsx) but NOT integrated
```

## Section 7: Preview and Pre-Download Infrastructure

### Current Preview Capabilities

**Status**: NO LIVE PREVIEW for certificates (fire safety type)

Users can:
1. ✅ Fill form with all data
2. ✅ Save/submit certificate (stored in DB)
3. ✅ Navigate to saved certificate detail page
4. ✅ Click "Download PDF" → PDF is **generated on-demand**
5. ❌ NO preview window before download (certificate must be saved first)

### Preview Infrastructure Found

#### 1. Report Disseminator PDF Preview (Different Feature)
**Location**: `components/disseminator/PdfPageCanvas.tsx`

This is used for the **Report Disseminator** feature (custom PDF form templates), NOT for fire safety certificates:
- Renders base64-encoded PDF to HTML canvas using **pdfjs-dist**
- Overlays interactive field selection using **react-konva**
- Allows users to define fields on a template by clicking regions
- **Not integrated into certificate workflow**

**Usage Pattern**:
```typescript
<PdfPageCanvas 
  pdfBase64={templatePdf}      // Base64 PDF
  pageNumber={1}               // Which page
  fields={fieldOverlays}       // Field bounding boxes
  selectedId={selected}        // For highlighting
  onSelectField={handleSelect} // Click handler
/>
```

#### 2. Download Button (Current Entry Point)
**Location**: `components/DownloadPDFButton.tsx`

```typescript
const handleDownload = async (e: React.MouseEvent) => {
  const response = await fetch(`/api/certificates/${certificateId}/pdf`);
  const blob = await response.blob();
  // Browser download...
}
```

- Simple fetch → blob → download
- No preview

#### 3. Certificate Details Page (View Before Download)
**Location**: `app/(dashboard)/certificates/[certificateId]/page.tsx`

Shows:
- Certificate metadata (number, type, status)
- Customer info
- Site details
- Certificate items (as list view, not formatted as PDF)
- **Download PDF button**

### Opportunity for Live Preview

**Potential Preview Implementation**:
```
1. Form fills in real-time (while editing)
   ↓
2. Optional "Preview PDF" button (generates without saving)
   ↓
3. Opens modal with PDF rendering in iframe or canvas
   ↓
4. User reviews, then "Save & Download" or "Make Changes"
```

**Technical Requirements**:
- Add new API route: `/api/certificates/preview` (accepts JSON, returns PDF)
- Use `PdfPageCanvas` component or iframe to display PDF
- Currently: Preview PHP possible but not implemented

### No Staging / Temporary Generation Path
- PDFs are generated only on-demand when `/api/certificates/[id]/pdf` is called
- No caching, no staged PDFs, no temporary file storage
- Each download regenerates PDF from live DB data

## Section 6: Field Transformations Between Form & PDF

### Common Transformations Applied

| Field | Form Input → PDF Output | Code Location |
|-------|----------|---|
| `inspectionDate` | ISO string (YYYY-MM-DD) | `formatDate()` → "DD Month YYYY" | `lib/pdf/generator.ts` |
| `nextInspectionDate` | ISO string | `formatDate()` → "DD Month YYYY" | `lib/pdf/generator.ts` |
| `formData.earthingArrangement` | "TN-C-S" | Kept as-is in table row | `generateEICRPDF()` |
| `boolean` fields (Yes/No) | Form select/radio | Rendered as text string | `lib/pdf/generator.ts` |
| `observations[]` | Array of {description, code} | Rendered with colored boxes (C1=red, C2=orange, C3=blue, FI=purple) | `generateEICRPDF()` lines 925-1000 |
| `certificateItems[]` | Array with {itemType, location, description, status, defects, recommendations} | Rendered as table + defect sections | `generateCertificatePDF()` lines 215-280 |
| Long text | No wrapping in form | Auto-wrapped in PDF using `pdf.splitTextToSize()` | `lib/pdf/generator.ts` |
| Multi-line text | Textarea | Displayed over multiple lines with adjusted cell height | `generateEICRPDF()` row() function |

### Specific EICR Transformations

1. **Observations Mapping**:
   - Form: `observations[].code` in [C1, C2, C3, FI]
   - PDF: Map to color + label + status:
     ```
     C1 → red background, "C1 – Danger Present", status = unsatisfactory
     C2 → orange background, "C2 – Potentially Dangerous", status = unsatisfactory
     C3 → blue background, "C3 – Improvement Recommended", status = satisfactory
     FI → purple background, "FI – Further Investigation Required", status = not_tested
     ```

2. **Special Character Support**:
   - Input: "I\\"n" or "I dn" or "ohms"
   - PDF: Rendered as Δ (delta) or Ω (omega) using Symbol font
   - Example for electrical values: `"I\u0394n"`, `"\u03A9"`

3. **Conditional Rendering**:
   - If `evidenceOfAdditions == "No"`: skip "Estimated Age of Additions" field
   - If `overallAssessment == "SATISFACTORY"`: render green "NO DEFECTS" section
   - If defects exist: render red colored defect box for each item

### Form Fields NOT in PDF

Some form inputs are collected but not rendered:
- `Next Inspection calculation logic` (form has 1yr/3yr/5yr/10yr dropdown, PDF just shows final date)
- Some `formData` fields are UI-only or for internal tracking

### PDF Content NOT in Form

Some PDF content is generated automatically or from database:
- Company branding (from formData.tradingTitle, formData.companyEmail)
- Page headers/footers with company name and reference number
- Standard certification statements (hardcoded legal text)
- Signature placeholders

## Section 5: Template System Architecture

### Two-Tier Template Approach

#### Tier 1: Certificate Templates (`certificateTemplates` Table)
**Location**: `lib/db/schema.ts` (table `certificate_templates`)

```
certificateTemplates {
  id: serial (primary key)
  teamId: integer (FK to teams)
  name: varchar(255)
  certificateType: varchar(50)  // BS5839-1, BS5839-6, EICR, etc.
  isDefault: boolean
  isActive: boolean
  template: json                 // Full template config
  description: text
  version: integer
  createdAt: timestamp
  updatedAt: timestamp
  createdBy: integer (FK to users)
}
```

**Template JSON Schema** (from `lib/pdf/generator.ts`):
```typescript
interface TemplateConfig {
  colors: {
    primary: string;       // hex e.g. '#1a3a5c'
    secondary: string;
    accent: string;
    background: string;
    text: string;
  };
  fonts?: {
    heading: string;
    body: string;
    size: { small: number; medium: number; large: number };
  };
  layout?: {
    margins: { top: number; right: number; bottom: number; left: number };
    spacing: number;
  };
}
```

#### Tier 2: Hardcoded Layouts (in PDF Generator)
- Each certificate type has hardcoded section layouts
- Default colors, fonts, spacing defined as constants
- TemplateConfig from DB **overrides** defaults (if available)

### Template Persistence & Lookup

From `app/api/certificates/[id]/pdf/route.ts`:
```typescript
// Attempt to load team's active template for this certificate type
const templates = await db
  .select()
  .from(certificateTemplates)
  .where(
    and(
      eq(certificateTemplates.certificateType, certificate.certificateType),
      eq(certificateTemplates.isActive, true)
    )
  )
  .orderBy(certificateTemplates.createdAt)
  .limit(1);

if (templates.length > 0 && templates[0].template) {
  templateConfig = templates[0].template;  // Loaded and passed to generator
}
```

### Seed / Initialization

`scripts/seed-eicr-template.ts` demonstrates **dynamic template definitions**:
- Defines EICR template structure as sections array
- Each section has: id, type, label, visible, order, config
- Section types: header, title, certificate-number, data-table, text-block, etc.
- This template would be inserted into DB for a team

### Current Usage Pattern

**Hardcoded (Default)**
- All certificate rendering logic is **hardcoded** in `lib/pdf/generator.ts`
- No database lookup occurs unless explicitly implemented
- Default colors: navy [26, 58, 92], gold [255, 193, 7], etc.

**Dynamic (Optional)**
- Templates stored in `certificate_templates` table (exists but may not be fully integrated)
- When loaded, overrides default colors/fonts
- Seed script shows intended structure for dynamic definitions

## Section 4: Certificate Type Field Mappings

### BS5839-1 (Fire Detection & Alarm - Non-Domestic)

**User Form Fields** (`app/(dashboard)/certificates/new/bs5839-1/page.tsx`):
- certificateNumber (auto-generated: `BS5839-1-YYYYMMDD-NNN`)
- customerId (dropdown)
- siteName (auto-populated from customer)
- siteAddress (auto-populated from customer address)
- inspectionDate (defaults to today)
- nextInspectionDate
- inspectorName
- inspectorQualification
- systemType (L1, L2, L3)
- numberOfZones
- numberOfDevices
- controlPanelMake, controlPanelModel
- inspectionType
- (Plus guided mode for 13+ fields)

**PDF Output Sections** (from `lib/pdf/generator.ts` → `getSystemDetails()`):
1. Site Details
2. System/Equipment Details (System Type, Category, Panel Make/Model, Zones, Devices, Floors, Floor Area)
3. Inspection Details (Date, Inspector Name, Qualification, Type, Next Due, Status)
4. Equipment/Items Tested (from certificateItems table)
5. Defects and Recommendations (color-coded: green=satisfactory, red=unsatisfactory)
6. Certification Statement (standard legal text)
7. Signature Section (Inspector & Client)

---

### BS5839-6 (Fire Detection & Alarm - Domestic)

**User Form Fields**:
- certificateNumber (auto-generated)
- customerId
- siteName (property address)
- propertyType (Domestic, Bungalow, Flat, etc.)
- numberOfFloors
- inspectionDate
- nextInspectionDate
- gradeOfSystem (Grade D, E, F)
- numberOfSmokeSensors, numberOfHeatSensors, numberOfCOSensors
- interconnectionMethod
- powerSupply
- inspectorName, inspectorQualification

**PDF Sections**:
1. Site Details
2. System Details (Grade, Property Type, Detector counts, Interconnection, Power)
3. Inspection Details
4. Equipment Tested
5. Defects/Recommendations
6. Certification Statement
7. Signature Section

---

### EICR (Electrical Installation Condition Report)

**User Form Fields** (`app/(dashboard)/certificates/new/eicr/page.tsx`):
- certificateNumber (CE format)
- customerId
- siteName (Client/Organisation name)
- clientAddress (Person ordering the report)
- inspectionDate (today by default)
- nextInspectionDate (calculated from dropdown: 1/3/5/10 years)
- reasonForReport
- installationAddress, premisesType (Domestic/Commercial/Industrial)
- estimatedAgeOfWiring, estimatedAgeOfAdditions
- evidenceOfAdditions (Yes/No)
- installationRecordsAvailable
- dateOfLastInspection
- extentOfInspection
- agreedLimitations, agreedLimitationsWith
- operationalLimitations
- earthingArrangement (TN-C-S, TN-S, IT, TT)
- meansOfEarthing
- overallAssessment (SATISFACTORY, C1, C2, C3)
- observations[] (array of {description, code: C1|C2|C3|FI})
- generalCondition
- tradingTitle, companyEmail, registrationNumber
- inspectorName, inspectorQualification

**PDF Sections** (8-page document per `generateEICRPDF()`):
1. Cover Page: Report titles, reference standards (BS 7671)
2. Section 1: Person Ordering the Report (Client name, address)
3. Section 2: Reason for Report (reason, inspection dates)
4. Section 3: Installation Details (address, wiring age, additions, records, last inspection)
5. Section 4: Extent & Limitations (what was covered, agreed limitations, operational limits)
6. Section 5: Assessment (earthing arrangement, means of earthing disposition)
7. Section 6-7: Test Results & Observations (color-coded by severity: C1=red, C2=orange, C3=blue, FI=purple)
8. Section 8: Declaration (trading title, installer details, signatures)

Each observation is rendered with:
- Severity code (C1: Danger, C2: Potentially Dangerous, C3: Improvement Recommended, FI: Further Investigation)
- Description
- Color-coded background box

---

### BS5266 (Emergency Lighting)

**User Form Fields**:
- certificateNumber
- customerId
- siteName
- systemType (Maintained, Non-maintained, Mixed)
- numberOfLamps, numberOfBatteryBlocks
- testingDate
- nextTestDate
- inspectorName

**PDF Sections**:
- Site information
- System type and equipment count
- Inspection results (pass/fail tests)
- Signature block

---

### FIRE_EXTINGUISHER (Portable Fire Extinguisher)

**User Form Fields**:
- certificateNumber
- customerId
- siteName, siteAddress
- inventory (textarea: extinguisher details)
- serviceDate
- nextServiceDate

**PDF Sections**:
- Inventory table
- Service details

---

### DRY_RISER (Dry Riser System)

**User Form Fields**:
- certificateNumber
- customerId
- siteName
- buildingHeight (meters)
- numberOfInlets
- testingDate
- nextTestDate
- workRequired
- recommendations
- certifierSignature

**PDF Sections**:
- Building information
- Test results
- Work required
- Inspector signature

## Section 3: Form → PDF Data Flow (Complete Pipeline)

### Step-by-Step Form Submission → PDF Rendering

```
[1. User Form Entry]
    ↓
    Form pages (app/(dashboard)/certificates/new/[certType]/page.tsx)
    - Collects certificate metadata (number, date, inspector)
    - Collects certificate items (inspections, defects, recommendations)
    - Collects formData (JSON) for type-specific fields
    
[2. Form Submission Handler]
    ↓
    onSubmit / handleSubmit calls createCertificate()
    - Parses FormData object
    - Separates main certificate fields from formData
    - Serializes certificate items if present
    
[3. Server Action (app/(dashboard)/actions.ts)]
    ↓
    createCertificate() receives parsed data:
    {
      customerId,
      certificateType,
      certificateNumber,
      siteName,
      siteAddress,
      inspectionDate,
      nextInspectionDate,
      inspectorName,
      formData: {...},          // Type-specific fields as JSON object
      items: [...]              // Certificate items array
    }
    
[4. Database Insert]
    ↓
    Insert into DB:
    - certificates table (main certificate record with formData as JSON column)
    - certificateItems table (one row per item with defects, recommendations, etc.)
    
[5. PDF Generation Trigger]
    ↓
    User clicks "Download PDF" → calls /api/certificates/[id]/pdf
    
[6. PDF API Route (app/api/certificates/[id]/pdf/route.ts)]
    ↓
    Fetches from DB:
    - certificates record (with formData JSON)
    - customer record (linked via customerId)
    - certificateItems (linked via certificateId)
    - Optional: certificateTemplates (for color/font override)
    
    Constructs CertificateData object:
    {
      id, certificateNumber, certificateType, siteName, siteAddress,
      inspectionDate, nextInspectionDate, inspectorName, status, formData,
      templateConfig: { colors, fonts, layout },
      customer: { name, email, phone, address, postcode, contactPerson },
      items: [{ itemType, location, description, status, defects, recommendations }, ...]
    }

[7. PDF Generation (lib/pdf/generator.ts)]
    ↓
    generateCertificatePDF(certificateData):
    - Routes EICR → generateEICRPDF()
    - Routes others → generic fire safety template
    - Uses formData for type-specific sections
    - Uses items[] for equipment/defect tables
    - Uses templateConfig for branding

[8. PDF Output]
    ↓
    Returns Uint8Array
    ↓ HTTP Response (Content-Type: application/pdf)
    ↓ Browser Downloads file
```

### Key Data Transformations

| Stage | Input | Output | Transformation |
|-------|-------|--------|-----------------|
| Form → Action | FormData object | Parsed TypeScript object | Extract and parse fields |
| Action → DB | TypeScript object | JSON stored in formData column | `.toJSON()` on complex fields |
| DB → PDF Generator | Certificate + Items + Template | CertificateData interface | Merge DB records with metadata |
| PDF Generator → Rendering | formData JSON + Items | Rendered sections | Format dates, concat addresses, map enums to labels |

## Section 2: PDF Generation Architecture

### Main Entry Point: `lib/pdf/generator.ts`

The PDF generator exports:
- **`generateCertificatePDF(certificate: CertificateData): Uint8Array`** - Main export that routes EICR to dedicated generator
- **`generateEICRPDF(certificate: CertificateData): Uint8Array`** - Specialized EICR generator (8-page document)
- **`TemplateConfig` interface** - Defines color schemes, fonts, and layout configuration
- **`CertificateData` interface** - Defines the complete data model passed to the generator

### Routing Logic
```
generateCertificatePDF()
├─ if certificateType === 'EICR'
│  └─→ generateEICRPDF() [8-page specialized layout]
│
└─ else
   └─→ Generic fire safety certificate generator
      ├─ BS5839-1
      ├─ BS5839-6
      ├─ BS5266
      ├─ FIRE_EXTINGUISHER
      └─ DRY_RISER
```

### Key Features
- **Template-aware rendering**: Optionally loads `TemplateConfig` from database to customize colors/fonts
- **Color system**: Supports primary, secondary, accent colors for customization
- **Font support**: Dynamic Greek letter (Δ, Ω) support for electrical values
- **Multi-page with headers/footers**: Automatic page breaks, page numbers, company branding
- **Conditional rendering**: Different layouts based on certificate type and form data

## Section 1: Key File Paths for PDF/Template Generation

### Primary Files
| Component | File Path | Purpose |
|-----------|-----------|---------|
| PDF Generator | `lib/pdf/generator.ts` | Main PDF generation engine |
| Certificate Schema | `lib/db/schema.ts` | Database tables: certificates, certificateItems, certificateTemplates |
| Seed Template | `scripts/seed-eicr-template.ts` | EICR template definitions (dynamic template system) |
| API Route | `app/api/certificates/[id]/pdf/route.ts` | REST endpoint for PDF generation |
| Download Component | `components/DownloadPDFButton.tsx` | Client-side PDF download trigger |
| PDF Canvas Preview | `components/disseminator/PdfPageCanvas.tsx` | Canvas-based PDF rendering (used by report disseminator) |

### Certificate Type Form Routes
- `app/(dashboard)/certificates/new/bs5839-1/page.tsx`
- `app/(dashboard)/certificates/new/bs5839-6/page.tsx`
- `app/(dashboard)/certificates/new/bs5266/page.tsx`
- `app/(dashboard)/certificates/new/fire-extinguisher/page.tsx`
- `app/(dashboard)/certificates/new/dry-riser/page.tsx`
- `app/(dashboard)/certificates/new/eicr/page.tsx`

# AI Certify Codebase Exploration: Certificate/PDF Generation System

## Exploration Scope
This notebook comprehensively explores the ai_certify codebase to understand:
- Certificate PDF generation logic and architecture
- Template system (hardcoded vs. dynamic)
- Form-to-PDF data flow and field transformations
- Preview capabilities for certificate generation
- Field inventory for each certificate type (BS5839-1, BS5839-6, EICR, etc.)

**Repository Root**: `/Users/admin/Development/ai_certify`